In [111]:
import pandas as pd
import numpy as np

data = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
!wget $data -O car_fuel_efficiency.csv
data = pd.read_csv("car_fuel_efficiency.csv")
df = data[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
features = ['engine_displacement','horsepower','vehicle_weight','model_year']
target = 'fuel_efficiency_mpg'


--2025-10-06 15:37:12--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv’

car_fuel_efficiency 100%[===================>] 853.70K  --.-KB/s    in 0.02s   

2025-10-06 15:37:12 (45.7 MB/s) - ‘car_fuel_efficiency.csv’ saved [874188/874188]



In [112]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X_b = np.column_stack([ones, X])
    XTX = X_b.T.dot(X_b)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X_b.T).dot(y)
    w0 = w_full[0]
    w = w_full[1:]
    return w0, w

def train_ridge_regression(X, y, r):
    ones = np.ones(X.shape[0])
    X_b = np.column_stack([ones, X])
    n_features = X_b.shape[1]
    I = np.eye(n_features)
    I[0,0] = 0
    w_full = np.linalg.inv(X_b.T.dot(X_b) + r*I).dot(X_b.T).dot(y)
    w0 = w_full[0]
    w = w_full[1:]
    return w0, w

def predict_linear(X, w0, w):
    return w0 + X.dot(w)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))


In [113]:
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df))
n = len(df)
n_train = int(0.6*n)
n_val = int(0.2*n)
n_test = n - n_train - n_val

train_idx = shuffled_indices[:n_train]
val_idx = shuffled_indices[n_train:n_train+n_val]
test_idx = shuffled_indices[n_train+n_val:]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)


In [114]:
X_train_0 = df_train[features].fillna(0).values
X_val_0 = df_val[features].fillna(0).values
y_train = df_train[target].values
y_val = df_val[target].values

mean_train = df_train[features].mean()
X_train_mean = df_train[features].fillna(mean_train).values
X_val_mean = df_val[features].fillna(mean_train).values

w0_0, w_0 = train_linear_regression(X_train_0, y_train)
y_pred_0 = predict_linear(X_val_0, w0_0, w_0)
rmse_0 = round(rmse(y_val, y_pred_0), 2)

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)
y_pred_mean = predict_linear(X_val_mean, w0_mean, w_mean)
rmse_mean = round(rmse(y_val, y_pred_mean), 2)

rmse_0, rmse_mean


(0.52, 0.46)

In [115]:
X_train = df_train[features].fillna(0).values
X_val = df_val[features].fillna(0).values
y_train = df_train[target].values
y_val = df_val[target].values

r_list = [0, 0.01, 0.1, 1, 5, 10, 100]
best_rmse = float('inf')
best_r = None

for r in r_list:
    w0, w = train_ridge_regression(X_train, y_train, r)
    y_pred = predict_linear(X_val, w0, w)
    score = round(rmse(y_val, y_pred), 2)
    if score < best_rmse:
        best_rmse = score
        best_r = r
    elif score == best_rmse:
        best_r = min(best_r, r)

best_r, best_rmse


(0, 0.52)

In [116]:
seeds = [0,1,2,3,4,5,6,7,8,9]
rmse_list = []

for seed in seeds:
    np.random.seed(seed)
    shuffled_indices = np.random.permutation(len(df))
    n_train = int(0.6*n)
    n_val = int(0.2*n)
    train_idx = shuffled_indices[:n_train]
    val_idx = shuffled_indices[n_train:n_train+n_val]
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_val = df.iloc[val_idx].reset_index(drop=True)

    X_train = df_train[features].fillna(0).values
    y_train = df_train[target].values
    X_val = df_val[features].fillna(0).values
    y_val = df_val[target].values

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = predict_linear(X_val, w0, w)
    rmse_list.append(rmse(y_val, y_pred))

std_rmse = round(np.std(rmse_list), 3)
std_rmse


0.007

In [117]:
np.random.seed(9)
shuffled_indices = np.random.permutation(len(df))
n_train = int(0.6*n)
n_val = int(0.2*n)
train_idx = shuffled_indices[:n_train]
val_idx = shuffled_indices[n_train:n_train+n_val]
test_idx = shuffled_indices[n_train+n_val:]

trainval_idx = np.concatenate([train_idx, val_idx])
df_trainval = df.iloc[trainval_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

X_trainval = df_trainval[features].fillna(0).values
y_trainval = df_trainval[target].values
X_test = df_test[features].fillna(0).values
y_test = df_test[target].values

r = 0.001
w0, w = train_ridge_regression(X_trainval, y_trainval, r)
y_pred_test = predict_linear(X_test, w0, w)
rmse_test = round(rmse(y_test, y_pred_test), 3)
rmse_test


0.515